# Cocoa pod ripeness segmentation with YOLO26 nano

This notebook converts the RipSetCocoaCNCH12 COCO annotations to Ultralytics YOLO segmentation format, then fine-tunes pretrained `yolo26n-seg.pt`.

The dataset has five ripeness-related classes: stage 1, stage 2, stage 3, harvest, and aborted pods. Download/extract the dataset from [Zenodo](https://zenodo.org/records/7968315) first; the archive is large (~27 GB).

In [1]:
%pip install -q -U ultralytics

import json, random, shutil, os
from pathlib import Path
from collections import defaultdict, Counter
import yaml
from PIL import Image
from ultralytics import YOLO

print('Ultralytics ready')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics ready


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/CocoaData

!aria2c \
  -x 16 \
  -s 16 \
  -k 1M \
  --continue=true \
  --file-allocation=none \
  --summary-interval=10 \
  "https://zenodo.org/records/7968315/files/RipSetCocoaCNCH12.zip?download=1" \
  -o RipSetCocoaCNCH12.zip

/content/drive/MyDrive/CocoaData

11/02 21:16:32 [NOTICE] Downloading 1 item(s)
 *** Download Progress Summary as of Mon Nov  2 21:16:42 2026 *** 
=
[#3ac84b 153MiB/25GiB(0%) CN:16 DL:11MiB ETA:37m33s]
FILE: /content/drive/MyDrive/CocoaData/RipSetCocoaCNCH12.zip
-

 *** Download Progress Summary as of Mon Nov  2 21:16:52 2026 *** 
=
[#3ac84b 306MiB/25GiB(1%) CN:16 DL:15MiB ETA:27m26s]
FILE: /content/drive/MyDrive/CocoaData/RipSetCocoaCNCH12.zip
-

 *** Download Progress Summary as of Mon Nov  2 21:17:03 2026 *** 
=
[#3ac84b 473MiB/25GiB(1%) CN:16 DL:15MiB ETA:27m3s]
FILE: /content/drive/MyDrive/CocoaData/RipSetCocoaCNCH12.zip
-

 *** Download Progress Summary as of Mon Nov  2 21:17:13 2026 *** 
=
[#3ac84b 662MiB/25GiB(2%) CN:16 DL:19MiB ETA:21m56s]
FILE: /content/drive/MyDrive/CocoaData/RipSetCocoaCNCH12.zip
-

 *** Download Progress Summary as of Mon Nov  2 21:17:24 2026 *** 
=
[#3ac84b 864MiB/25GiB(3%) CN:16 DL:20MiB ETA:20m38s]
FILE: /content/drive/MyDrive/CocoaData/RipSetCocoaCNCH1

In [ ]:
!unzip -q \
  /content/drive/MyDrive/CocoaData/RipSetCocoaCNCH12.zip \
  -d /content/drive/MyDrive/CocoaData/

In [ ]:
# ---- EDIT THESE OPTIONS ----
DATASET_ROOT = Path('/content/drive/MyDrive/CocoaData/RipSetCocoaCNCH12')  # extracted Zenodo folder
OUT = Path('/content/drive/MyDrive/CocoaData/cocoa_yolo26')
SEED = 42
SAMPLE_FRACTION = 0.05       # 5% smoke test; set to 1.0 for the full dataset
TRAIN_FRACTION, VAL_FRACTION = 0.80, 0.10

# Training options
MODEL = 'yolo26n-seg.pt'       # nano segmentation checkpoint
EPOCHS = 50
IMGSZ = 640
BATCH = 8                     # lower this if GPU memory is insufficient
DEVICE = 0                    # use 'cpu' or 0, 1, ...
WORKERS = 2
PATIENCE = 15

# Set True to discard the 'aborted' class and train only on four ripeness stages
DROP_ABORTED = False

assert DATASET_ROOT.exists(), f'Extract the dataset first: {DATASET_ROOT}'

In [ ]:
# Find the COCO annotation file and image folder automatically.
json_files = list(DATASET_ROOT.rglob('*.json'))
print('JSON files:', [str(p.relative_to(DATASET_ROOT)) for p in json_files[:10]])

annotation_file = next((p for p in json_files if 'coco' in p.name.lower() or 'annotation' in p.name.lower()), json_files[0])
coco = json.loads(annotation_file.read_text())
categories = coco['categories']
print('Using:', annotation_file)
print('Categories:', [(c['id'], c['name']) for c in categories])

# Index all image files by filename.
image_files = {}
for p in DATASET_ROOT.rglob('*'):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        image_files.setdefault(p.name, p)
print('Images found:', len(image_files))

In [ ]:
# Convert COCO polygons to YOLO segmentation labels and split by source image.
random.seed(SEED)
cat_ids = [c['id'] for c in categories]
cat_names = [c['name'] for c in categories]
if DROP_ABORTED:
    keep = [i for i, n in enumerate(cat_names) if 'abort' not in n.lower()]
    cat_ids = [cat_ids[i] for i in keep]
    cat_names = [cat_names[i] for i in keep]
cat_to_yolo = {cid: i for i, cid in enumerate(cat_ids)}

anns_by_image = defaultdict(list)
for ann in coco['annotations']:
    if ann['category_id'] in cat_to_yolo and ann.get('segmentation'):
        anns_by_image[ann['image_id']].append(ann)

images = [im for im in coco['images'] if im['id'] in anns_by_image and im['file_name'].split('/')[-1] in image_files]
random.shuffle(images)
if SAMPLE_FRACTION < 1.0:
    images = images[:max(1, int(len(images) * SAMPLE_FRACTION))]
print('Images selected:', len(images))
n_train = int(len(images) * TRAIN_FRACTION)
n_val = int(len(images) * VAL_FRACTION)
splits = {'train': images[:n_train], 'val': images[n_train:n_train+n_val], 'test': images[n_train+n_val:]}

if OUT.exists():
    shutil.rmtree(OUT)
for split, split_images in splits.items():
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    for im in split_images:
        src = image_files[im['file_name'].split('/')[-1]]
        dst = OUT / 'images' / split / src.name
        shutil.copy2(src, dst)
        lines = []
        W, H = im['width'], im['height']
        for ann in anns_by_image[im['id']]:
            seg = ann['segmentation']
            if not isinstance(seg, list):
                continue  # skip RLE masks in this small converter
            for polygon in seg:
                if len(polygon) < 6: continue
                pts = [str(x / W if j % 2 == 0 else x / H) for j, x in enumerate(polygon)]
                lines.append(str(cat_to_yolo[ann['category_id']]) + ' ' + ' '.join(pts))
        (OUT / 'labels' / split / f'{src.stem}.txt').write_text('\n'.join(lines))

(OUT / 'cocoa.yaml').write_text(yaml.safe_dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': dict(enumerate(cat_names))}, sort_keys=False))
print({k: len(v) for k, v in splits.items()})
print('Classes:', cat_names)
print((OUT / 'cocoa.yaml').read_text())

In [ ]:
# Transfer learning: pretrained ImageNet/COCO features, new cocoa segmentation head.
model = YOLO(MODEL)
results = model.train(
    data=str(OUT / 'cocoa.yaml'),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    pretrained=True,
    project=str(OUT / 'runs'),
    name='ripeness_seg',
    plots=True,
)

In [ ]:
# Validate and run one prediction.
best = OUT / 'runs' / 'ripeness_seg' / 'weights' / 'best.pt'
trained = YOLO(str(best))
metrics = trained.val(data=str(OUT / 'cocoa.yaml'), imgsz=IMGSZ, batch=BATCH, device=DEVICE)
print('mAP50-95:', metrics.seg.map)

sample = next((OUT / 'images' / 'test').glob('*'))
trained.predict(source=str(sample), conf=0.25, save=True, save_txt=True)
print('Prediction:', sample)

## Useful options

- `SAMPLE_FRACTION = 0.05` runs a quick 5% smoke test; use `1.0` for all images.
- `DROP_ABORTED = True` trains only the four ripeness stages.
- Reduce `BATCH` if CUDA runs out of memory; reduce `IMGSZ` to 512 for a faster experiment.
- Increase `EPOCHS` to 100–200 for a serious run.
- For a CPU-only test, set `DEVICE = 'cpu'`, but training will be much slower.
- The converter skips COCO RLE masks; it uses polygon segmentation annotations, which are the usual format for this dataset.